# Bronze: Flights from Kafka
- **Source**: Event Hubs topic `flights-raw`
- **Target**: Delta table at `abfss://bronze@.../flights`

Reads ALL messages from the topic (earliest), writes to Delta **without parsing the JSON payload**. Silver will do the parsing.
This notebook uses **Spark Structured Streaming with `Trigger.AvailableNow`**, which processes all currently-available Kafka data and stops — perfect for batch-style orchestration via Airflow.

## Params

In [0]:
dbutils.widgets.text("mode", "backfill", "backfill | incremental")
dbutils.widgets.text("starting_offset", "earliest", "earliest | latest")

MODE = dbutils.widgets.get("mode")
STARTING_OFFSET = dbutils.widgets.get("starting_offset")

print(f"Running in mode={MODE} with starting_offset={STARTING_OFFSET}")

## Imports and Config

In [0]:
email = dbutils.secrets.get(scope="flight-delay", key="db-email")

In [0]:
import sys
sys.path.insert(0, f"/Workspace/Users/{email}/ml-final-project/databricks")

from common.config import (
    BRONZE_FLIGHTS_PATH,
    CHECKPOINT_FLIGHTS,
    TOPIC_FLIGHTS,
    get_kafka_options,
    get_secret,
    configure_storage_auth,
)

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

## Authentication to ADLS

In [0]:
configure_storage_auth(spark)

## Read data from Event Hub - Kafka

In [0]:
eh_connection_string = get_secret("eventhubs-consumer-connection-string")

kafka_options = get_kafka_options(
    topic=TOPIC_FLIGHTS,
    connection_string=eh_connection_string,
    starting_offset=STARTING_OFFSET,
)

raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

print("Kafka source schema:")
raw_stream.printSchema()

## Transformation: Minimal Bronze Layer Transforms

In [0]:
bronze_df = (
    raw_stream
    .select(
        F.col("key").cast(StringType()).alias("kafka_key"),
        F.col("value").cast(StringType()).alias("kafka_value"),
        F.col("topic").alias("kafka_topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.current_timestamp().alias("ingestion_ts_utc"),
        F.lit(MODE).alias("ingestion_mode"),
    )
)

## Write to Delta Tables

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_FLIGHTS)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .start(BRONZE_FLIGHTS_PATH)
)

# Block until the micro-batch finishes
query.awaitTermination()

print("Bronze flights write complete.")

### Register as Delta Table

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS bronze.flights
    USING DELTA
    LOCATION '{BRONZE_FLIGHTS_PATH}'
""")

### Test the Delta Table

In [0]:
row_count = spark.read.format("delta").load(BRONZE_FLIGHTS_PATH).count()
print(f"Total rows in bronze.flights: {row_count:,}")

## Exit

In [0]:
dbutils.notebook.exit(f'{{"rows_in_bronze": {row_count}}}')